# Compare promptfoo runs

Reads a snapshot of the promptfoo DB; nothing is written back. Edit the config cell, then run all cells top to bottom.

In [1]:
# Config: which runs to compare
LAST_N_RUNS = 2  # compare the newest N runs (runs with no results are skipped)
TITLE_FILTER = None  # optional regex on run titles, case-insensitive, applied before LAST_N_RUNS
EVAL_IDS = []  # optional explicit ids, e.g. ["eval-1EM-2026-10-14T14:00:56"]; overrides the two above
DB_PATH = None  # None = $PROMPTFOO_CONFIG_DIR/promptfoo.db or ~/.promptfoo/promptfoo.db

In [2]:
# Pick up edits to compare_lib.py without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys

try:
    import pandas as pd
except ModuleNotFoundError:
    raise SystemExit(
        f"Wrong kernel: {sys.executable} has no pandas. Select the kernel "
        "'Python (promptfoo analysis)' (analysis/.venv/bin/python) and rerun."
    ) from None

import compare_lib as cl

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

conn = cl.snapshot(DB_PATH)

## Runs being compared

In [3]:
runs = cl.select_runs(conn, eval_ids=EVAL_IDS or None, title_filter=TITLE_FILTER, last_n=LAST_N_RUNS)
runs

,eval_id,created_at,title,n_results,models
0,eval-Jgq-2026-10-14T14:55:56,2026-10-14 16:55:56.299,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0]",4,qwen3-vl:4b
1,eval-1EM-2026-10-14T14:00:56,2026-10-14 16:00:56.716,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0]",4,qwen3-vl:4b


## Load results

In [4]:
df = cl.load_results(conn, eval_ids=runs["eval_id"].tolist())
print(f"{len(df)} results, {df['eval_id'].nunique()} runs, {df['column_id'].nunique()} columns, {df['test_key'].nunique()} tests, {int(df['is_error'].sum())} errors")
df.head()

8 results, 2 runs, 4 columns, 2 tests, 0 errors


,eval_id,title,created_at,column_id,prompt_idx,prompt_label,provider_id,model,provider_label,test_idx,test_key,success,score,failure_reason,is_error,error,cost,latency_ms,tokens_prompt,tokens_completion,tokens_total,output,cfg_num_predict,cfg_seed,cfg_temperature,cfg_think,cfg_showThinking,var_question,var_expected
0,eval-1EM-2026-10-14T14:00:56,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0] [1EM]",2026-10-14 16:00:56.716,1EM p0 id=qwen3-vl:4b temperature=0.3,0,Answer with only the number: {{question}},ollama:chat:qwen3-vl:4b,qwen3-vl:4b,id=qwen3-vl:4b temperature=0.3,0,b35745f371e1,True,1.0,0,False,None,NaN,3712,80,327,407,18,2048,512,0.3,True,False,Janet’s ducks lay 16 eggs per day. She eats three for breakfast every mornin...,18
1,eval-1EM-2026-10-14T14:00:56,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0] [1EM]",2026-10-14 16:00:56.716,1EM p0 id=qwen3-vl:4b temperature=0.3,0,Answer with only the number: {{question}},ollama:chat:qwen3-vl:4b,qwen3-vl:4b,id=qwen3-vl:4b temperature=0.3,1,b193c2a41a47,True,1.0,0,False,None,NaN,10571,42,240,282,3,2048,512,0.3,True,False,A robe takes 2 bolts of blue fiber and half that much white fiber. How many...,3
2,eval-1EM-2026-10-14T14:00:56,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0] [1EM]",2026-10-14 16:00:56.716,1EM p1 id=qwen3-vl:4b temperature=0.6,1,Answer with only the number: {{question}},ollama:chat:qwen3-vl:4b,qwen3-vl:4b,id=qwen3-vl:4b temperature=0.6,0,b35745f371e1,True,1.0,0,False,None,NaN,7931,78,391,469,18,2048,512,0.6,False,False,Janet’s ducks lay 16 eggs per day. She eats three for breakfast every mornin...,18
3,eval-1EM-2026-10-14T14:00:56,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0] [1EM]",2026-10-14 16:00:56.716,1EM p1 id=qwen3-vl:4b temperature=0.6,1,Answer with only the number: {{question}},ollama:chat:qwen3-vl:4b,qwen3-vl:4b,id=qwen3-vl:4b temperature=0.6,1,b193c2a41a47,True,1.0,0,False,None,NaN,12559,40,190,230,3,2048,512,0.6,False,False,A robe takes 2 bolts of blue fiber and half that much white fiber. How many...,3
4,eval-Jgq-2026-10-14T14:55:56,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0] [Jgq]",2026-10-14 16:55:56.299,Jgq p0 id=qwen3-vl:4b temperature=0.3,0,Answer with only the number: {{question}},ollama:chat:qwen3-vl:4b,qwen3-vl:4b,id=qwen3-vl:4b temperature=0.3,0,b35745f371e1,True,1.0,0,False,None,NaN,5904,80,327,407,18,2048,512,0.3,True,False,Janet’s ducks lay 16 eggs per day. She eats three for breakfast every mornin...,18


## Columns and the factors that vary between them

In [5]:
factors = cl.factor_columns(df)
print("factors:", factors)
df.groupby("column_id", sort=False)[["title", *factors]].first()

factors: ['cfg_temperature', 'cfg_think']


,title,cfg_temperature,cfg_think
column_id,,,
1EM p0 id=qwen3-vl:4b temperature=0.3,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0] [1EM]",0.3,True
1EM p1 id=qwen3-vl:4b temperature=0.6,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0] [1EM]",0.6,False
Jgq p0 id=qwen3-vl:4b temperature=0.3,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0] [Jgq]",0.3,True
Jgq p1 id=qwen3-vl:4b temperature=0.6,"GSM8k - sweep id [qwen3-vl:4b, qwen3.5:4b] x temperature [0.3, 0.6, 1.0] [Jgq]",0.6,False
